In [1]:
import os

REPO = 'giomamaca/walmart-sales-forecasting'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    from google.colab import userdata, files

    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
        print('Upload your kaggle.json')
        uploaded = files.upload()
        shutil.move('kaggle.json', f'{kaggle_dir}/kaggle.json')
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)

    github_token = userdata.get('GITHUB_TOKEN')
    repo_dir = f'/content/{REPO.split("/")[1]}'
    if not os.path.exists(repo_dir):
        !git clone -q https://{github_token}@github.com/{REPO}.git {repo_dir}
    os.chdir(repo_dir)

    !pip install -q kaggle mlflow wandb lightgbm

    import wandb
    wandb.login(key=userdata.get('WANDB_API_KEY'))

    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
    !unzip -o -q data/walmart-recruiting-store-sales-forecasting.zip -d data
    !for f in data/*.zip; do unzip -o -q "$f" -d data; done

    !mlflow db upgrade sqlite:///mlflow.db

    DATA_DIR = 'data'
else:
    DATA_DIR = '..'

Upload your kaggle.json


Saving kaggle.json to kaggle.json
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 110.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adola23 (adola23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100% 2.70M/2.70M [00:00<00:00, 195MB/s]

2026/07/11 04:57:47 INFO mlflow.store.db.utils: Updating database tables


In [2]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import wandb
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from lightgbm import LGBMRegressor

FIXED = dict(verbose=-1)

In [3]:
train = pd.read_csv(f'{DATA_DIR}/train.csv', parse_dates=['Date'])
features = pd.read_csv(f'{DATA_DIR}/features.csv', parse_dates=['Date'])
stores = pd.read_csv(f'{DATA_DIR}/stores.csv')

train.shape, features.shape, stores.shape

((421570, 5), (8190, 12), (45, 3))

In [4]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.average(np.abs(y_true - y_pred), weights=weights)

In [5]:
MARKDOWN_COLS = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
PRE_CHRISTMAS = pd.to_datetime(['2010-12-24', '2011-12-23', '2012-12-21'])

FEATURE_COLS = [
    'Store', 'Dept', 'IsHoliday', 'Size', 'Type',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
    'Year', 'Month', 'Week', 'IsPreChristmas',
]

class FeatureBuilder(BaseEstimator, TransformerMixin):
    def __init__(self, stores_df, features_df):
        self.stores_df = stores_df
        self.features_df = features_df

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.merge(self.stores_df, on='Store', how='left')
        df = df.merge(self.features_df.drop(columns=['IsHoliday']), on=['Store', 'Date'], how='left')

        df[MARKDOWN_COLS] = df[MARKDOWN_COLS].fillna(0)
        df['Type'] = df['Type'].map({'A': 0, 'B': 1, 'C': 2})
        df['IsHoliday'] = df['IsHoliday'].astype(int)

        df['Year'] = df['Date'].dt.year
        df['Month'] = df['Date'].dt.month
        df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
        df['IsPreChristmas'] = df['Date'].isin(PRE_CHRISTMAS).astype(int)

        return df[FEATURE_COLS]

In [6]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('LightGBM_v2_Training')

2026/07/11 04:57:59 INFO mlflow.tracking.fluent: Experiment with name 'LightGBM_v2_Training' does not exist. Creating a new experiment.


<Experiment: artifact_location='/content/walmart-sales-forecasting/mlruns/6', creation_time=1783745879901, effective_trace_archival_retention=None, experiment_id='6', last_update_time=1783745879901, lifecycle_stage='active', name='LightGBM_v2_Training', tags={}, trace_location=None, workspace='default'>

In [7]:
with mlflow.start_run(run_name='LightGBM_v2_Cleaning'):
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_v2_Cleaning', reinit='finish_previous')

    n_negative = int((train['Weekly_Sales'] < 0).sum())
    missing_markdown_pct = float(features[MARKDOWN_COLS].isna().mean().mean())

    mlflow.log_param('negative_sales_rows', n_negative)
    mlflow.log_param('negative_sales_action', 'clip_to_zero')
    mlflow.log_metric('missing_markdown_pct', missing_markdown_pct)
    wandb.log({'negative_sales_rows': n_negative, 'missing_markdown_pct': missing_markdown_pct})
    wandb.finish()

    y_train = train['Weekly_Sales'].clip(lower=0)

missing_markdown_pct,▁
negative_sales_rows,▁
missing_markdown_pct,0.55849
negative_sales_rows,1285


In [8]:
def time_based_splits(dates, n_splits=3):
    dates = np.sort(dates.unique())
    fold_size = len(dates) // (n_splits + 1)
    splits = []
    for i in range(1, n_splits + 1):
        train_end = dates[fold_size * i - 1]
        val_end = dates[min(fold_size * (i + 1) - 1, len(dates) - 1)]
        splits.append((train_end, val_end))
    return splits

splits = time_based_splits(train['Date'])
splits

[(np.datetime64('2010-10-01T00:00:00.000000000'),
  np.datetime64('2011-06-03T00:00:00.000000000')),
 (np.datetime64('2011-06-03T00:00:00.000000000'),
  np.datetime64('2012-02-03T00:00:00.000000000')),
 (np.datetime64('2012-02-03T00:00:00.000000000'),
  np.datetime64('2012-10-05T00:00:00.000000000'))]

In [9]:
SEARCH_SPACE = {
    'n_estimators': [500, 1000, 1500],
    'num_leaves': [31, 63, 127, 255],
    'learning_rate': [0.03, 0.05, 0.1],
    'min_child_samples': [20, 50, 100],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
}
N_TRIALS = 15

rng = np.random.default_rng(42)

def sample_params(rng):
    return {k: v[rng.integers(len(v))] for k, v in SEARCH_SPACE.items()}

builder = FeatureBuilder(stores, features)
fold_data = []
for train_end, val_end in splits:
    tm = train['Date'] <= train_end
    vm = (train['Date'] > train_end) & (train['Date'] <= val_end)
    fold_data.append((
        builder.transform(train[tm]), y_train[tm],
        builder.transform(train[vm]), y_train[vm],
        train.loc[vm, 'IsHoliday'].values,
    ))

def cv_score(params):
    scores = []
    for X_tr, y_tr, X_val, y_val, hol in fold_data:
        model = LGBMRegressor(**params, **FIXED)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        scores.append(wmae(y_val.values, preds, hol))
    return scores

with mlflow.start_run(run_name='LightGBM_v2_Tuning'):
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_v2_Tuning', reinit='finish_previous')

    best_wmae, best_params = None, None
    for t in range(N_TRIALS):
        params = sample_params(rng)
        scores = cv_score(params)
        mean = float(np.mean(scores))
        mlflow.log_metric('trial_wmae_mean', mean, step=t)
        wandb.log({'trial': t, 'trial_wmae_mean': mean, **params})
        if best_wmae is None or mean < best_wmae:
            best_wmae, best_params = mean, params
        print(t, round(mean, 1), params)

    mlflow.log_metric('best_wmae_mean', best_wmae)
    mlflow.log_dict(best_params, 'best_params.json')
    wandb.log({'best_wmae_mean': best_wmae})
    wandb.finish()

best_wmae, best_params

0 3049.7 {'n_estimators': 500, 'num_leaves': 255, 'learning_rate': 0.05, 'min_child_samples': 50, 'subsample': 0.8, 'colsample_bytree': 1.0}
1 3321.5 {'n_estimators': 500, 'num_leaves': 127, 'learning_rate': 0.03, 'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 1.0}
2 3079.6 {'n_estimators': 1500, 'num_leaves': 255, 'learning_rate': 0.1, 'min_child_samples': 100, 'subsample': 0.8, 'colsample_bytree': 0.7}
3 3262.4 {'n_estimators': 1500, 'num_leaves': 63, 'learning_rate': 0.05, 'min_child_samples': 50, 'subsample': 0.7, 'colsample_bytree': 1.0}
4 3104.4 {'n_estimators': 1500, 'num_leaves': 127, 'learning_rate': 0.05, 'min_child_samples': 100, 'subsample': 0.8, 'colsample_bytree': 0.8}
5 4067.4 {'n_estimators': 1000, 'num_leaves': 31, 'learning_rate': 0.03, 'min_child_samples': 50, 'subsample': 1.0, 'colsample_bytree': 0.7}
6 2969.6 {'n_estimators': 1500, 'num_leaves': 255, 'learning_rate': 0.03, 'min_child_samples': 50, 'subsample': 0.7, 'colsample_bytree': 1.0}
7 3445.9 

best_wmae_mean,▁
colsample_bytree,██▁█▃▁██▃█▁▁▁██
learning_rate,▃▁█▃▃▁▁▁█▃▃▁██▁
min_child_samples,▄▁█▄█▄▄█▁▁█▄▄▄▁
n_estimators,▁▁███▅███▅██▅▅▅
num_leaves,█▄█▂▄▁█▂█▁▄▂▁▄▄
subsample,▃▃▃▁▃█▁▃▃█▃█▁▁█
trial,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
trial_wmae_mean,▂▃▂▃▂█▁▄▁▅▂▄▄▂▂
best_wmae_mean,2917.76307
colsample_bytree,1


(2917.763069440465,
 {'n_estimators': 1500,
  'num_leaves': 255,
  'learning_rate': 0.1,
  'min_child_samples': 20,
  'subsample': 0.8,
  'colsample_bytree': 0.8})

In [10]:
with mlflow.start_run(run_name='LightGBM_v2_CV'):
    mlflow.log_params(best_params)
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_v2_CV', config=best_params, reinit='finish_previous')

    fold_scores = cv_score(best_params)
    for i, s in enumerate(fold_scores):
        mlflow.log_metric('wmae', s, step=i)
        wandb.log({'fold': i, 'wmae': s})

    mlflow.log_metric('wmae_mean', float(np.mean(fold_scores)))
    mlflow.log_metric('wmae_std', float(np.std(fold_scores)))
    wandb.log({'wmae_mean': float(np.mean(fold_scores)), 'wmae_std': float(np.std(fold_scores))})
    wandb.finish()

fold_scores

fold,▁▅█
wmae,█▄▁
wmae_mean,▁
wmae_std,▁
fold,2
wmae,2027.6571
wmae_mean,2917.76307
wmae_std,783.75476


[np.float64(3934.822008501629),
 np.float64(2790.810103356151),
 np.float64(2027.6570964636157)]

In [11]:
with mlflow.start_run(run_name='LightGBM_v2_Final'):
    mlflow.log_params(best_params)
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_v2_Final', config=best_params, reinit='finish_previous')

    pipeline = Pipeline([
        ('features', FeatureBuilder(stores, features)),
        ('model', LGBMRegressor(**best_params, **FIXED)),
    ])
    pipeline.fit(train[['Store', 'Dept', 'Date', 'IsHoliday']], y_train)

    preds = pipeline.predict(train[['Store', 'Dept', 'Date', 'IsHoliday']])
    train_wmae = wmae(y_train.values, preds, train['IsHoliday'].values)
    mlflow.log_metric('train_wmae', train_wmae)
    wandb.log({'train_wmae': train_wmae})

    mlflow.sklearn.log_model(pipeline, name='model', serialization_format='cloudpickle')
    wandb.finish()

train_wmae

2026/07/11 05:17:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


train_wmae,▁
train_wmae,944.09949


np.float64(944.0994940175301)

In [12]:
if IN_COLAB:
    import requests
    gh_user = requests.get('https://api.github.com/user', headers={'Authorization': f'token {github_token}'}).json()['login']
    !git config user.name "{gh_user}"
    !git config user.email "{gh_user}@users.noreply.github.com"

    !git add mlflow.db mlruns
    !git commit -m "Run model_experiment_LightGBM_v2.ipynb in Colab"
    !git push

[main 3c31454] Run model_experiment_LightGBM_v2.ipynb in Colab
 7 files changed, 83 insertions(+)
 create mode 100644 mlruns/6/23595badaacc4d2d9b6f7322c9375663/artifacts/best_params.json
 create mode 100644 mlruns/6/models/m-4547cf1b67fd4a32973ae41e74fa831c/artifacts/MLmodel
 create mode 100644 mlruns/6/models/m-4547cf1b67fd4a32973ae41e74fa831c/artifacts/conda.yaml
 create mode 100644 mlruns/6/models/m-4547cf1b67fd4a32973ae41e74fa831c/artifacts/model.pkl
 create mode 100644 mlruns/6/models/m-4547cf1b67fd4a32973ae41e74fa831c/artifacts/python_env.yaml
 create mode 100644 mlruns/6/models/m-4547cf1b67fd4a32973ae41e74fa831c/artifacts/requirements.txt
Enumerating objects: 19, done.
Counting objects: 100% (19/19), done.
Delta compression using up to 2 threads
Compressing objects: 100% (13/13), done.
Writing objects: 100% (16/16), 11.47 MiB | 4.93 MiB/s, done.
Total 16 (delta 3), reused 3 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://